In [1]:
"""
Flujo de trabajo de segmentación de imágenes científicas
Este notebook integra las principales herramientas de segmentación presentadas hasta ahora en un flujo de trabajo completo de análisis de imágenes.
En lugar de estudiar por separado el threshold, la morphology, los componentes conectados y las mediciones de región, ahora los combinamos en un flujo de trabajo práctico que comienza con una imagen en escala de grises y termina con un análisis cuantitativo basado en objetos.
Objetivos de aprendizaje
- Construir un workflow de segmentación completo
- Aplicar el threshold, la morphology y el análisis de componentes conectados de forma secuencial
- Medir cuantitativamente los objetos segmentados
- Visualizar los resultados intermedios y finales del workflow
- Comprender la segmentación como un proceso de análisis reproducible
Contexto de la imagen reales
En la imagen reales, la segmentación rara vez es una sola operación. Generalmente es un workflow compuesto por:
- Preparación de la imagen
- Segmentación inicial
- Limpieza de la máscara
- Separación de objetos
- Medición cuantitativa
Este cuaderno muestra cómo estas etapas funcionan juntas de forma reproducible.
"""

'\nFlujo de trabajo de segmentación de imágenes científicas\nEste notebook integra las principales herramientas de segmentación presentadas hasta ahora en un flujo de trabajo completo de análisis de imágenes.\nEn lugar de estudiar por separado el threshold, la morphology, los componentes conectados y las mediciones de región, ahora los combinamos en un flujo de trabajo práctico que comienza con una imagen en escala de grises y termina con un análisis cuantitativo basado en objetos.\nObjetivos de aprendizaje\n- Construir un workflow de segmentación completo\n- Aplicar el threshold, la morphology y el análisis de componentes conectados de forma secuencial\n- Medir cuantitativamente los objetos segmentados\n- Visualizar los resultados intermedios y finales del workflow\n- Comprender la segmentación como un proceso de análisis reproducible\nContexto de la imagen reales\nEn la imagen reales, la segmentación rara vez es una sola operación. Generalmente es un workflow compuesto por:\n- Prepar

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import data, filters, morphology, measure, color, segmentation, util

In [3]:
def show_image(image, title="", cmap="gray", save_path=None):
    plt.figure(figsize=(6, 6))
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.show()


def compare_images(images, titles, cmap="gray", figsize=(15, 5), save_path=None):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)

    if len(images) == 1:
        axes = [axes]

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.show()


def save_dataframe_csv(df, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)